In [6]:
# !conda install pytorch torchvision torchaudio -c pytorch-nightly
# !pip install torch transformers accelerate datasets scikit-learn numpy matplotlib tqdm

import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA
from collections import defaultdict

###############################################################################
# Arguments
###############################################################################
args = {
    "models": ["nlpaueb/sec-bert-base", "bert-base-uncased"],
    "datasets": [
        {
            "name": "yelp_review_full",
            "config": None,
            "split": "train",
            "text_column": "text",
        },
        {
            "name": "wikitext",
            "config": "wikitext-2-raw-v1",
            "split": "train",
            "text_column": "text",
        },
        {"name": "ag_news", "config": None, "split": "train", "text_column": "text"},
    ],
    "max_texts": 1000,
    "batch_size": 64,
    "drift_strengths": [0.0, 0.25, 0.5, 0.75, 1.0],
    "pca_components": 2,
    "output_dir": "results_multidataset",
}
os.makedirs(args["output_dir"], exist_ok=True)


###############################################################################
# Utility Functions
###############################################################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]


def extract_cls_embeddings(model, tokenizer, texts, device):
    encodings = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=128
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()


def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    import random

    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            new_texts.append(txt)
            continue
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]
        new_texts.append(" ".join(words))
    return new_texts


###############################################################################
# Additional Vector/Feature-Space Distance Functions
###############################################################################
def euclidean_distance(x, y):
    """
    Euclidean (L2) Distance

    Formula:
        d_{L2}(x, y) = sqrt( sum_i (x_i - y_i)^2 )
    """
    return np.sqrt(np.sum((x - y) ** 2))


def manhattan_distance(x, y):
    """
    Manhattan (L1) Distance

    Formula:
        d_{L1}(x, y) = sum_i |x_i - y_i|
    """
    return np.sum(np.abs(x - y))


def minkowski_distance(x, y, p=3):
    """
    Minkowski Distance (generalization of L1 and L2)

    Formula:
        d_{p}(x, y) = ( sum_i (|x_i - y_i|^p) )^(1/p)

    - p=1 -> Manhattan distance
    - p=2 -> Euclidean distance
    """
    return np.sum(np.abs(x - y) ** p) ** (1.0 / p)


def chebyshev_distance(x, y):
    """
    Chebyshev (L∞) Distance

    Formula:
        d_{∞}(x, y) = max_i( |x_i - y_i| )
    """
    return np.max(np.abs(x - y))


def correlation_distance(x, y):
    """
    Correlation Distance

    Formula:
        d_corr(x, y) = 1 - corr(x, y)

    where corr(x, y) is the Pearson correlation coefficient:
        corr(x, y) = ( (x - mean(x)) · (y - mean(y)) )
                     / ( ||x - mean(x)|| * ||y - mean(y)|| )

    Note: This distance ignores differences in magnitude and focuses on the
          correlation between x and y.
    """
    x_centered = x - np.mean(x)
    y_centered = y - np.mean(y)
    numerator = np.dot(x_centered, y_centered)
    denominator = (np.linalg.norm(x_centered) * np.linalg.norm(y_centered)) + 1e-12
    corr = numerator / denominator
    return 1.0 - corr


def canberra_distance(x, y):
    """
    Canberra Distance

    Formula:
        d_canberra(x, y) = sum_i( |x_i - y_i| / (|x_i| + |y_i|) )

    More sensitive to differences when x_i or y_i are near zero.
    """
    numerator = np.abs(x - y)
    denominator = np.abs(x) + np.abs(y) + 1e-12
    return np.sum(numerator / denominator)


###############################################################################
# We collect all "point-to-point" distance functions in a dictionary
# so we can iterate over them easily:
###############################################################################
DISTANCE_FUNCTIONS = {
    "euclidean": euclidean_distance,
    "manhattan": manhattan_distance,
    "minkowski": lambda u, v: minkowski_distance(u, v, p=3),  # example with p=3
    "chebyshev": chebyshev_distance,
    "correlation": correlation_distance,
    "canberra": canberra_distance,
}


###############################################################################
# Unified Drift Detection Class:
# - It can do Mahalanobis distance or any of the simpler distances.
###############################################################################
class EmbeddingTracker:
    """
    A tracker that maintains:
    - An exponential-moving-average (EMA) of the embedding mean
    - An exponential-moving-average covariance (for Mahalanobis)
    - A selected distance metric: either "mahalanobis" or one from DISTANCE_FUNCTIONS
    """

    def __init__(self, embedding_dim, alpha=0.01, distance_name="mahalanobis"):
        self.alpha = alpha
        self.distance_name = distance_name

        # For all trackers, keep track of the mean.
        self.mean = np.zeros((embedding_dim,))
        self.count = 0

        # Mahalanobis-specific
        self.cov = np.eye(embedding_dim)

    def update(self, embedding):
        """Update the running mean (and covariance, if Mahalanobis)."""
        if self.count == 0:
            self.mean = embedding
            if self.distance_name == "mahalanobis":
                self.cov = np.eye(len(embedding))
        else:
            self.mean = (1 - self.alpha) * self.mean + self.alpha * embedding

            if self.distance_name == "mahalanobis":
                # Update covariance for Mahalanobis
                diff = embedding - self.mean
                self.cov = (1 - self.alpha) * self.cov + self.alpha * np.outer(
                    diff, diff
                )
        self.count += 1

    def compute_distance(self, embedding):
        """Compute distance between 'embedding' and the tracker's current mean."""
        if self.distance_name == "mahalanobis":
            diff = embedding - self.mean
            cov_inv = np.linalg.pinv(self.cov)
            return np.sqrt(diff.T @ cov_inv @ diff)
        else:
            # Use the dictionary of distance functions
            dist_fn = DISTANCE_FUNCTIONS[self.distance_name]
            return dist_fn(self.mean, embedding)


###############################################################################
# Main Data Collection
###############################################################################
def collect_data():
    # Detect device
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    results = defaultdict(list)

    for dataset_info in args["datasets"]:
        dataset_name = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col = dataset_info["text_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = list(ds[text_col])
        random.shuffle(texts)
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]

        half_point = len(texts) // 2
        baseline_texts = texts[:half_point]
        drift_texts = texts[half_point:]

        for model_name in args["models"]:
            print(f"\n--- Using Model: {model_name} ---")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name)
            model.to(device)
            model.eval()

            # Precompute baseline embeddings
            baseline_embs = []
            for b in batch_generator(baseline_texts, args["batch_size"]):
                emb_b = extract_cls_embeddings(model, tokenizer, b, device)
                baseline_embs.append(emb_b)
            baseline_embs = np.concatenate(baseline_embs, axis=0)

            # Fit PCA on the baseline embeddings
            pca = PCA(n_components=args["pca_components"])
            pca.fit(baseline_embs)

            key = (dataset_name, model_name)

            # We'll try multiple distances, including Mahalanobis
            all_distance_names = ["mahalanobis"] + list(DISTANCE_FUNCTIONS.keys())

            for distance_name in all_distance_names:
                for drift_strength in args["drift_strengths"]:
                    print(
                        f"Distance={distance_name}, drift_strength={drift_strength} ..."
                    )
                    drifted_texts = introduce_gradual_drift(
                        drift_texts, fraction_shuffle=drift_strength
                    )
                    test_texts = baseline_texts + drifted_texts

                    # 1) No PCA
                    embedding_dim_no_pca = baseline_embs.shape[1]
                    tracker_no_pca = EmbeddingTracker(
                        embedding_dim_no_pca, alpha=0.01, distance_name=distance_name
                    )
                    distance_scores_no_pca = []
                    all_embeddings_no_pca = []

                    # Initialize tracker with baseline mean embeddings
                    for batch in batch_generator(baseline_texts, args["batch_size"]):
                        emb = extract_cls_embeddings(model, tokenizer, batch, device)
                        mean_emb = emb.mean(axis=0)
                        tracker_no_pca.update(mean_emb)

                    # Now process the test set, batch by batch
                    for batch in tqdm(
                        batch_generator(test_texts, args["batch_size"]), leave=False
                    ):
                        emb = extract_cls_embeddings(model, tokenizer, batch, device)
                        all_embeddings_no_pca.extend(emb)
                        mean_emb = emb.mean(axis=0)
                        dist = tracker_no_pca.compute_distance(mean_emb)
                        distance_scores_no_pca.append(dist)
                        tracker_no_pca.update(mean_emb)

                    final_dist_no_pca = (
                        distance_scores_no_pca[-1] if distance_scores_no_pca else 0.0
                    )

                    # 2) With PCA
                    embedding_dim_pca = args["pca_components"]
                    tracker_pca = EmbeddingTracker(
                        embedding_dim_pca, alpha=0.01, distance_name=distance_name
                    )
                    distance_scores_pca = []
                    all_embeddings_pca = []

                    # Initialize tracker with baseline (reduced)
                    for batch in batch_generator(baseline_texts, args["batch_size"]):
                        emb = extract_cls_embeddings(model, tokenizer, batch, device)
                        emb_pca = pca.transform(emb)
                        mean_emb = emb_pca.mean(axis=0)
                        tracker_pca.update(mean_emb)

                    for batch in tqdm(
                        batch_generator(test_texts, args["batch_size"]), leave=False
                    ):
                        emb = extract_cls_embeddings(model, tokenizer, batch, device)
                        emb_pca = pca.transform(emb)
                        all_embeddings_pca.extend(emb_pca)
                        mean_emb = emb_pca.mean(axis=0)
                        dist = tracker_pca.compute_distance(mean_emb)
                        distance_scores_pca.append(dist)
                        tracker_pca.update(mean_emb)

                    final_dist_pca = (
                        distance_scores_pca[-1] if distance_scores_pca else 0.0
                    )

                    # Store results
                    results[key].append(
                        {
                            "distance_name": distance_name,
                            "drift_strength": drift_strength,
                            "pca": False,
                            "time_series": distance_scores_no_pca[:],
                            "final_similarity": final_dist_no_pca,
                            "all_embeddings": np.array(all_embeddings_no_pca),
                        }
                    )
                    results[key].append(
                        {
                            "distance_name": distance_name,
                            "drift_strength": drift_strength,
                            "pca": True,
                            "time_series": distance_scores_pca[:],
                            "final_similarity": final_dist_pca,
                            "all_embeddings": np.array(all_embeddings_pca),
                        }
                    )

    print("\nData collection done!")
    return results


###############################################################################
# Run Experiments
###############################################################################
all_results = collect_data()

Using device: mps

=== Loading dataset: yelp_review_full ===

--- Using Model: nlpaueb/sec-bert-base ---
Distance=mahalanobis, drift_strength=0.0 ...


Distance=mahalanobis, drift_strength=0.25 ...


Distance=mahalanobis, drift_strength=0.5 ...


Distance=mahalanobis, drift_strength=0.75 ...


Distance=mahalanobis, drift_strength=1.0 ...


...

Distance=canberra, drift_strength=0.25 ...


Distance=canberra, drift_strength=0.5 ...


Distance=canberra, drift_strength=0.75 ...


Distance=canberra, drift_strength=1.0 ...



Data collection done!


In [1]:
import os
import math


def plot_distance_vs_drift_strength(all_results):
    """
    For each (dataset_name, model_name) in all_results, we create ONE figure
    that has multiple subplots—one for each distance_name. Each subplot shows
    two lines: “No PCA” vs “PCA.”
    """

    for (dataset_name, model_name), runs in all_results.items():
        # Gather all distance names that were used
        distance_names = sorted(list({r["distance_name"] for r in runs}))

        n_dists = len(distance_names)

        # Decide on a grid layout (e.g. 3 columns)
        n_cols = 3
        n_rows = math.ceil(n_dists / n_cols)

        # Create figure and axes
        fig, axs = plt.subplots(
            n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharex=True
        )
        # Flatten the axes in case we have multiple rows
        axs = axs.flatten()

        # Figure title
        fig.suptitle(f"{dataset_name} | {model_name}", fontsize=16)

        # Plot each distance_name in its own subplot
        for i, dist_name in enumerate(distance_names):
            ax = axs[i]

            # Filter runs for this distance_name & No PCA
            no_pca_runs = [
                r for r in runs if r["distance_name"] == dist_name and not r["pca"]
            ]
            no_pca_runs = sorted(no_pca_runs, key=lambda x: x["drift_strength"])

            # Filter runs for this distance_name & PCA
            pca_runs = [r for r in runs if r["distance_name"] == dist_name and r["pca"]]
            pca_runs = sorted(pca_runs, key=lambda x: x["drift_strength"])

            # Extract x=drift_strength and y=final_distance
            x_no_pca = [run["drift_strength"] for run in no_pca_runs]
            y_no_pca = [run["final_similarity"] for run in no_pca_runs]
            x_pca = [run["drift_strength"] for run in pca_runs]
            y_pca = [run["final_similarity"] for run in pca_runs]

            # Plot lines for No PCA vs PCA
            ax.plot(x_no_pca, y_no_pca, marker="o", label="No PCA")
            ax.plot(x_pca, y_pca, marker="s", label="PCA")

            ax.set_title(dist_name, fontsize=10)
            ax.set_xlabel("Drift Strength")
            ax.set_ylabel("Distance")
            ax.grid(True, linestyle="--", alpha=0.5)

            # Only add legend on first subplot or so to reduce clutter
            if i == 0:
                ax.legend()

        # If we have any leftover axes (e.g., when n_dists < n_rows * n_cols),
        # make them invisible
        for j in range(i + 1, n_rows * n_cols):
            axs[j].set_visible(False)

        # Adjust layout
        plt.tight_layout(rect=[0, 0, 1, 0.96])  # leave space for suptitle

        # Save figure
        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_distances_subplots.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path, dpi=300)
        plt.show()
        print(f"Saved figure: {save_path}")


plot_distance_vs_drift_strength(all_results)

NameError: name 'all_results' is not defined